In [ ]:
# import libraries
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix

In [ ]:
# Linking Google drive to use preprocessed data
from google.colab import drive

# This will prompt for authorization.
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

# Define the path to your Google Drive directory
ser_folder = "/content/drive/My Drive/SER"

# Create the directory if it doesn't exist
if not os.path.exists(ser_folder):
    os.makedirs(ser_folder)

In [ ]:
# Define the base directory for your dataset in Google Drive
base_dir = "/content/drive/My Drive/SER"  # Adjust the path according to your directory structure

# Define paths to train and test directories
train_dir = os.path.join(base_dir, "train")
test_dir = os.path.join(base_dir, "test")

# Verify the paths
print("Train directory:", train_dir)
print("Test directory:", test_dir)

Train directory: /content/drive/My Drive/SER/train
Test directory: /content/drive/My Drive/SER/test


In [ ]:
# create the folders

# --train--
folder = "/content/drive/My Drive/SER_std/train/angry"
if not os.path.exists(folder):
  os.makedirs(folder)

folder = "/content/drive/My Drive/SER_std/train/disgust"
if not os.path.exists(folder):
  os.makedirs(folder)

folder = "/content/drive/My Drive/SER_std/train/happy"
if not os.path.exists(folder):
  os.makedirs(folder)

folder = "/content/drive/My Drive/SER_std/train/fear"
if not os.path.exists(folder):
  os.makedirs(folder)

folder = "/content/drive/My Drive/SER_std/train/neutral"
if not os.path.exists(folder):
  os.makedirs(folder)

folder = "/content/drive/My Drive/SER_std/train/sad"
if not os.path.exists(folder):
  os.makedirs(folder)

folder = "/content/drive/My Drive/SER_std/train/surprise"
if not os.path.exists(folder):
  os.makedirs(folder)

# --test--

folder = "/content/drive/My Drive/SER_std/test/angry"
if not os.path.exists(folder):
  os.makedirs(folder)

folder = "/content/drive/My Drive/SER_std/test/disgust"
if not os.path.exists(folder):
  os.makedirs(folder)

folder = "/content/drive/My Drive/SER_std/test/happy"
if not os.path.exists(folder):
  os.makedirs(folder)

folder = "/content/drive/My Drive/SER_std/test/fear"
if not os.path.exists(folder):
  os.makedirs(folder)

folder = "/content/drive/My Drive/SER_std/test/neutral"
if not os.path.exists(folder):
  os.makedirs(folder)

folder = "/content/drive/My Drive/SER_std/test/sad"
if not os.path.exists(folder):
  os.makedirs(folder)

folder = "/content/drive/My Drive/SER_std/test/surprise"
if not os.path.exists(folder):
  os.makedirs(folder)

In [ ]:
""" Prepare the files """

# Unified audio segmentation
import os
from pydub import AudioSegment
import librosa
import torch
import numpy as np

""" standardize audio duration by cutting or padding and convert from 1d audio to 2d mel spectrogram tensor """
def prep_files(input_dir, output_dir, target_duration_s=3, n_mels=64):
    """
    Cuts or pads with silence all audio files in a directory to a target duration
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Supported extensions (pydub handles these if ffmpeg is installed)
    valid_extensions = ('.wav', '.m4a')

    for file_name in os.listdir(input_dir):
        if not file_name.lower().endswith(valid_extensions):
            continue

        file_path = os.path.join(input_dir, file_name)
        output_path = os.path.join(output_dir, file_name)

        try:
            # Load audio file
            audio, sr = librosa.load(file_path, sr=22050)
            current_duration = len(audio) / sr
            target_duration_s *= sr

            if current_duration > target_duration_s:
                # Cut the audio if it's too long
                processed_audio = audio[:target_duration_s]
            elif current_duration < target_duration_s:
                # Pad with silence if it's too short
                processed_audio = np.pad(audio, (0, target_duration_s - len(audio)), 'constant')
                # silence_needed = target_duration_s - current_duration
                # silence = AudioSegment.silent(duration=silence_needed, frame_rate=sr)
                # processed_audio = audio + silence
            else:
                processed_audio = audio

            # Convert 1D wave signal to 2D Mel Spectrogram
            mel_spec = librosa.feature.melspectrogram(
                y=processed_audio, sr=sr, n_mels=n_mels, n_fft=1024, hop_length=512
            )

            # Convert power to decibels (log scale matches human hearing)
            mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

            # Add a Channel dimension (1, n_mels, time_steps) to match CNN expectation
            # PyTorch CNNs expect: (Batch, Channel, Height, Width)
            audio_tensor = torch.tensor(mel_spec_db, dtype=torch.float32).unsqueeze(0)

            torch.save(audio_tensor, output_path)

        except Exception as e:
            print(f"Failed to process {file_name}: {e}")

# --- Configuration ---
TARGET_DURATION = 3

# --train--
INPUT_DATASET = "/content/drive/My Drive/SER/train/angry"
OUTPUT_DATASET = "/content/drive/My Drive/SER_std/train/angry"
prep_files(INPUT_DATASET, OUTPUT_DATASET, TARGET_DURATION)

INPUT_DATASET = "/content/drive/My Drive/SER/train/disgust"
OUTPUT_DATASET = "/content/drive/My Drive/SER_std/train/disgust"
prep_files(INPUT_DATASET, OUTPUT_DATASET, TARGET_DURATION)

INPUT_DATASET = "/content/drive/My Drive/SER/train/happy"
OUTPUT_DATASET = "/content/drive/My Drive/SER_std/train/happy"
prep_files(INPUT_DATASET, OUTPUT_DATASET, TARGET_DURATION)

INPUT_DATASET = "/content/drive/My Drive/SER/train/fear"
OUTPUT_DATASET = "/content/drive/My Drive/SER_std/train/fear"
prep_files(INPUT_DATASET, OUTPUT_DATASET, TARGET_DURATION)

INPUT_DATASET = "/content/drive/My Drive/SER/train/neutral"
OUTPUT_DATASET = "/content/drive/My Drive/SER_std/train/neutral"
prep_files(INPUT_DATASET, OUTPUT_DATASET, TARGET_DURATION)

INPUT_DATASET = "/content/drive/My Drive/SER/train/sad"
OUTPUT_DATASET = "/content/drive/My Drive/SER_std/train/sad"
prep_files(INPUT_DATASET, OUTPUT_DATASET, TARGET_DURATION)

INPUT_DATASET = "/content/drive/My Drive/SER/train/surprise"
OUTPUT_DATASET = "/content/drive/My Drive/SER_std/train/surprise"
prep_files(INPUT_DATASET, OUTPUT_DATASET, TARGET_DURATION)

# --test--

INPUT_DATASET = "/content/drive/My Drive/SER/test/angry"
OUTPUT_DATASET = "/content/drive/My Drive/SER_std/test/angry"
prep_files(INPUT_DATASET, OUTPUT_DATASET, TARGET_DURATION)

INPUT_DATASET = "/content/drive/My Drive/SER/test/disgust"
OUTPUT_DATASET = "/content/drive/My Drive/SER_std/test/disgust"
prep_files(INPUT_DATASET, OUTPUT_DATASET, TARGET_DURATION)

INPUT_DATASET = "/content/drive/My Drive/SER/test/happy"
OUTPUT_DATASET = "/content/drive/My Drive/SER_std/test/happy"
prep_files(INPUT_DATASET, OUTPUT_DATASET, TARGET_DURATION)

INPUT_DATASET = "/content/drive/My Drive/SER/test/fear"
OUTPUT_DATASET = "/content/drive/My Drive/SER_std/test/fear"
prep_files(INPUT_DATASET, OUTPUT_DATASET, TARGET_DURATION)

INPUT_DATASET = "/content/drive/My Drive/SER/test/neutral"
OUTPUT_DATASET = "/content/drive/My Drive/SER_std/test/neutral"
prep_files(INPUT_DATASET, OUTPUT_DATASET, TARGET_DURATION)

INPUT_DATASET = "/content/drive/My Drive/SER/test/sad"
OUTPUT_DATASET = "/content/drive/My Drive/SER_std/test/sad"
prep_files(INPUT_DATASET, OUTPUT_DATASET, TARGET_DURATION)

INPUT_DATASET = "/content/drive/My Drive/SER/test/surprise"
OUTPUT_DATASET = "/content/drive/My Drive/SER_std/test/surprise"
prep_files(INPUT_DATASET, OUTPUT_DATASET, TARGET_DURATION)

In [ ]:
from torch import nn

class CNNNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Sequential(
        nn.Conv2d(
            in_channels=1,
            out_channels=16,
            kernel_size=3,
            stride=1,
            padding=2,
        )
        bb.ReLU(),
        nn.MaxPool2d(kernel_size=2),
    )

    self.conv2 = nn.Sequential(
        nn.Conv2d(
            in_channels=1,
            out_channels=16,
            kernel_size=3,
            stride=1,
            padding=2,
        )
        bb.ReLU(),
        nn.MaxPool2d(kernel_size=2),
      )

    self.conv3 = nn.Sequential(
        nn.Conv2d(
            in_channels=1,
            out_channels=16,
            kernel_size=3,
            stride=1,
            padding=2,
        )
        bb.ReLU(),
        nn.MaxPool2d(kernel_size=2),
    )

    self.conv4 = nn.Sequential(
        nn.Conv2d(
            in_channels=1,
            out_channels=16,
            kernel_size=3,
            stride=1,
            padding=2,
        )
        bb.ReLU(),
        nn.MaxPool2d(kernel_size=2),
    )